# 01 Data Preprocessing and Visualisation

This notebook visualises dataset audit outputs and final split protocol.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent
TABLES_DIR = PROJECT_ROOT / 'outputs' / 'tables'
FIG_DIR = PROJECT_ROOT / 'outputs' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

audit_path = TABLES_DIR / 'dataset_audit.csv'
manifest_path = TABLES_DIR / 'split_manifest.csv'

if not audit_path.exists() or not manifest_path.exists():
    raise FileNotFoundError('Run `python main.py audit` before this notebook.')

audit_df = pd.read_csv(audit_path)
manifest_df = pd.read_csv(manifest_path)
audit_df.head()

In [ ]:
summary = (
    audit_df[~audit_df['broken_file_flag']]
    .groupby(['split_raw', 'class_name'])
    .size()
    .unstack(fill_value=0)
)
summary

In [ ]:
final_summary = (
    manifest_df.groupby(['split_final', 'class_name'])
    .size()
    .unstack(fill_value=0)
)
final_summary

In [ ]:
plt.figure(figsize=(9, 4))
plot_df = final_summary.reset_index().melt(id_vars='split_final', var_name='class_name', value_name='count')
sns.barplot(data=plot_df, x='split_final', y='count', hue='class_name')
plt.title('Class Distribution by Final Split')
plt.xlabel('Final split')
plt.ylabel('Image count')
plt.tight_layout()
plt.savefig(FIG_DIR / 'notebook_data_distribution.png', dpi=180)
plt.show()

In [ ]:
from PIL import Image
import numpy as np

classes = ['CNV', 'DME', 'DRUSEN', 'NORMAL']
subset = manifest_df[manifest_df['split_final'] == 'train_final']

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, cls in enumerate(classes):
    cls_rows = subset[subset['class_name'] == cls].head(2)
    for j, (_, row) in enumerate(cls_rows.iterrows()):
        img = np.array(Image.open(row['filepath']).convert('L'))
        ax = axes[j, i]
        ax.imshow(img, cmap='gray')
        ax.set_title(f'{cls} #{j+1}')
        ax.axis('off')

plt.tight_layout()
plt.savefig(FIG_DIR / 'notebook_sample_grid.png', dpi=180)
plt.show()